# تشغيل Qwen كنموذج خاص بك (يعمل مع الواجهة)
1. **Runtime ← Change runtime type ← T4 GPU** (مجاني).
2. شغّل الخلايا بالترتيب: 1 ← 2 ← 3 ← 4.
3. انسخ الرابط من الخلية 4 وضعه بالواجهة: الإعدادات ← **API متوافق مع OpenAI** ← رابط API. (المفتاح فاضي، اسم الموديل أي شي.)
4. الخلية 5 اختيارية: تدريب إضافي (LoRA) على بياناتك.

الرابط يتغير كل مرة تشغل الدفتر، وينقطع إذا سكّر Colab.

## 1) تثبيت المكتبات

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes peft fastapi uvicorn

## 2) تحميل النموذج
غيّر `MODEL` إذا تريد حجم ثاني: `Qwen/Qwen2.5-3B-Instruct` أخف وأسرع، و`Qwen/Qwen2.5-7B-Instruct` أذكى (مضغوط 4-bit ليدخل بـ T4).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-7B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)
model.eval()
print("جاهز:", MODEL)

## 3) سيرفر متوافق مع OpenAI (يرد تدريجياً)

In [ ]:
import json, threading, uvicorn
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from transformers import TextIteratorStreamer

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

def stream(messages, max_tokens, temperature):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    kw = dict(**inputs, streamer=streamer, max_new_tokens=max_tokens,
              do_sample=temperature > 0, temperature=max(temperature, 0.01), top_p=0.9)
    threading.Thread(target=model.generate, kwargs=kw, daemon=True).start()
    for piece in streamer:
        yield piece

@app.post("/v1/chat/completions")
async def chat(req: Request):
    b = await req.json()
    msgs = b.get("messages", [])
    def gen():
        for t in stream(msgs, min(int(b.get("max_tokens", 1024)), 2048), float(b.get("temperature", 0.7))):
            yield "data: " + json.dumps({"choices": [{"delta": {"content": t}}]}, ensure_ascii=False) + "\n\n"
        yield "data: [DONE]\n\n"
    return StreamingResponse(gen(), media_type="text/event-stream")

@app.get("/")
def root():
    return {"status": "ok", "model": MODEL}

threading.Thread(target=uvicorn.run, kwargs=dict(app=app, host="0.0.0.0", port=8000, log_level="warning"), daemon=True).start()
print("السيرفر شغال على المنفذ 8000")

## 4) فتح رابط عام (Cloudflare Tunnel، بدون حساب)

In [ ]:
import subprocess, re, time
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
p = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in p.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break
print("\nضع هذا الرابط بالواجهة (رابط API):\n" + url + "/v1/chat/completions")

## 5) (اختياري) تدريب إضافي LoRA على بياناتك
ارفع ملف `train.jsonl` (من قائمة الملفات يسار)، كل سطر محادثة:
```
{"messages":[{"role":"user","content":"سؤال"},{"role":"assistant","content":"الجواب"}]}
```
بعد انتهاء التدريب يستخدم السيرفر النموذج المعدّل مباشرة بدون إعادة تشغيل. ابدأ بـ 100 مثال أو أكثر، وخلّ الأمثلة متنوعة.

In [ ]:
import json, torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

rows = [json.loads(l) for l in open("train.jsonl", encoding="utf-8") if l.strip()]
ds = Dataset.from_list([{"text": tok.apply_chat_template(r["messages"], tokenize=False)} for r in rows])
ds = ds.map(lambda x: tok(x["text"], truncation=True, max_length=512), remove_columns=["text"])

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
model.print_trainable_parameters()

Trainer(model=model, train_dataset=ds,
    args=TrainingArguments(output_dir="out", per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=2, learning_rate=2e-4, fp16=True, logging_steps=5, save_strategy="no", report_to=[]),
    data_collator=DataCollatorForLanguageModeling(tok, mlm=False)).train()

model.eval()
model.save_pretrained("my-lora")   # حمّل مجلد my-lora إذا تريد تحفظه
print("خلص التدريب، النموذج المعدّل صار هو اللي يرد")